# 04 — Evaluation

This notebook evaluates the MLP translation head on the S3DIS dataset.

### 1. Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
BRANCH = 'main'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}


In [ ]:
%cd {REPO_DIR}
!git fetch origin
!git checkout -B {BRANCH} origin/{BRANCH}
!git pull origin {BRANCH}

In [ ]:
!git clone https://github.com/Pointcept/Concerto.git /content/Concerto 2>/dev/null || echo 'Concerto already cloned'

### 2. Symlink Data & Checkpoints

In [ ]:
!rm -f data checkpoints features pretrained
!ln -sf /content/drive/MyDrive/DL_Project/data ./data
!ln -sf /content/drive/MyDrive/DL_Project/checkpoints ./checkpoints
!ln -sf /content/drive/MyDrive/DL_Project/features ./features

### 3. Setup Environment

In [ ]:
%cd /content/Deep_learning_project/notebooks/
!uv python install 3.10.12
!uv sync --python 3.10.12

In [ ]:
# Setup HF token for open_clip
from google.colab import userdata
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    with open(".env", "w") as f:
        f.write(f"HF_TOKEN={HF_TOKEN}\n")
except userdata.SecretNotFoundError:
    print("No HF token found.")

### 4. Evaluate (OA & mIoU)

In [ ]:
%%time
CHECKPOINT = '/content/drive/MyDrive/DL_Project/checkpoints/mlp_s3dis/last_model.pth'
FEATURES_DIR = '/content/drive/MyDrive/DL_Project/features/s3dis_area5'

import os
if os.path.exists('/content/features_local/s3dis_area5'):
    FEATURES_DIR = '/content/features_local/s3dis_area5'

!PYTHONPATH=/content/Deep_learning_project:/content/Concerto \
  uv run python /content/Deep_learning_project/src/evaluate.py \
  --features_dir {FEATURES_DIR} \
  --checkpoint {CHECKPOINT} \
  --clip_model ViT-B-32 \
  --clip_pretrained openai